
# UK Biobank RAP Data Extraction

This notebook extracts relevant UK Biobank fields using the DNAnexus Research Analysis Platform (RAP). It builds a dataset for downstream phenotype derivation in R.

---

## Step 1: Import Required Packages

We use `pyspark` for distributed data processing, `dxpy` for DNAnexus API access, and `dxdata` for UKB RAP dataset handling.


In [ ]:

import pyspark
import dxpy
import dxdata



---

## Step 2: Initialize Spark

This sets up the Spark context and session. Run this only once per session.


In [ ]:

sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)



---

## Step 3: Discover UKB Dataset and Database

Automatically locate the dispensed UKB dataset and database in your DNAnexus project.


In [ ]:

dispensed_database = dxpy.find_one_data_object(
    classname='database',
    name='app*',
    folder='/',
    name_mode='glob',
    describe=True)
dispensed_database_name = dispensed_database['describe']['name']

dispensed_dataset = dxpy.find_one_data_object(
    typename='Dataset',
    name='app*.dataset',
    folder='/',
    name_mode='glob')
dispensed_dataset_id = dispensed_dataset['id']



---

## Step 4: Load Dataset

Load the UKB dataset using `dxdata`. This gives access to all entities (tables) including `participant`, `hesin`, `gp`, etc.


In [ ]:

dataset = dxdata.load_dataset(id=dispensed_dataset_id)
dataset.entities



---

## Step 5: Access Participant Entity

Most UKB fields are stored in the `participant` entity. We will extract fields from this table.


In [ ]:

participant = dataset['participant']



---

## Step 6: Define Field Lookup Functions

These helper functions allow you to find field names by UKB field ID or title keyword.


In [ ]:

from distutils.version import LooseVersion

def fields_for_id(field_id):
    field_id = str(field_id)
    fields = participant.find_fields(name_regex=r'^p{}(_i\d+)?(_a\d+)?$'.format(field_id))
    return sorted(fields, key=lambda f: LooseVersion(f.name))

def field_names_for_id(field_id):
    return [f.name for f in fields_for_id(field_id)]

def fields_by_title_keyword(keyword):
    fields = list(participant.find_fields(lambda f: keyword.lower() in f.title.lower()))
    return sorted(fields, key=lambda f: LooseVersion(f.name))

def field_names_by_title_keyword(keyword):
    return [f.name for f in fields_by_title_keyword(keyword)]



---

## Step 7: Define Fields to Extract

Identified the relevant UKB data fields from the old stat scripts and readme file, validated them on the Cohort viewer and Showcase and added them to the list below. 


In [ ]:

field_names = [
    'eid',           # Participant ID
    'p31',           # Sex
    'p2734_i0',      # Number of live births
    'p2774_i0',      # Ever had stillbirth, spontaneous miscarriage or termination 
    'p41202',        # HES main ICD-10 codes
    'p41203',        # HES main ICD9 codes
    'p30790_i0',     # Lipoprotein(a) measurements,
    'p30796_i0',     # Lipoprotein(a) reportability
    'p22000',        # Genotyping batch
    'p22020'         # Used in genetic principal components
    # Genetic principal components (1-40)
] + [f'p22009_a{i}' for i in range(1, 41)]



---

## Step 8: Retrieve Fields into Spark DataFrame

Use `retrieve_fields` to extract the selected fields into a Spark DataFrame.


In [ ]:

df = participant.retrieve_fields(names=field_names, engine=dxdata.connect())
df.show(5, truncate=False)
df.limit(5).toPandas()

## Step 8b: Filter for Females

Filter the Spark DataFrame to include only female participants (where sex field `p31` equals 0).


In [ ]:
df.count()

df_female = df.filter(df.p31 == 0)
df_female.show(5, truncate=False)


## Step 8c: Filter for Ever Pregnant Participants

Keep only females who had at least one pregnancy, defined as having at least one live birth (`p2734_i0 > 0`) or ever having a pregnancy complication (`p2774_i0 == 1`).


In [ ]:
df_ever_pregnant = df_female.filter((df_female.p2734_i0 > 0) & (df_female.p2734_i0.isNotNull()) | (df_female.p2774_i0 == 1))
df_ever_pregnant.show(5, truncate=False)

df_ever_pregnant.count()
#231169



---

## Step 9: Export Dataset

Save the extracted dataset as a `.tsv` file for downstream processing in R.


In [ ]:

df_ever_pregnant.toPandas().to_csv('extracted_data.tsv', sep='\t', index=False)



---

## Step 10: Upload to DNAnexus Project

Use the DNAnexus CLI to upload the file to your project folder.


In [ ]:

%%bash
dx upload extracted_data.tsv --dest data/UKB_RAP/



---

## Done

Proceed to the R scripts for dataset generation and cleaning.
